# Python · Excel, CSV, Parquet — учебная шпаргалка

Ноутбук с воспроизводимыми примерами из PDF-шпаргалки.
Все данные генерируются прямо в коде — внешние файлы не нужны.

**Дисциплина переменных:**

- `df_raw` — исходный набор, не изменяем
- `df_work` — рабочая копия для очистки и преобразований
- `df_clean` — финальный чистый набор для визуализации и сохранения

**Запуск:** `Kernel → Restart & Run All` — все ячейки выполнятся без ошибок.


## 0. Установка библиотек и проверка версий

Если какая-то библиотека не установлена, раскомментируйте нужную строку в следующей ячейке.


In [ ]:
# Установка библиотек (нужна только при первом запуске)
# В Jupyter и Colab используем ! перед pip; в терминале — без !

# !pip install pandas numpy openpyxl pyarrow matplotlib seaborn
# !pip install --upgrade pandas openpyxl pyarrow


In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import pyarrow
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("openpyxl:", openpyxl.__version__)
print("pyarrow:", pyarrow.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)

# Настройка визуализации
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11


## 1. Воспроизводимый демонстрационный набор данных

`sales_df` — синтетический датасет о заказах. Содержит пропуски и один дубликат
для демонстрации приёмов очистки.


In [ ]:
rng = np.random.default_rng(seed=42)

n = 30
sales_df = pd.DataFrame({
    "order_id": [f"ORD-{i+1:04d}" for i in range(n)],
    "order_date": pd.date_range(start="2024-01-01", periods=n, freq="D"),
    "client": rng.choice(["Alice", "Bob", "Charlie", "Diana", np.nan], size=n),
    "category": rng.choice(["Books", "Electronics", "Clothing", "Books"], size=n),
    "amount": np.round(rng.normal(loc=120, scale=40, size=n), 2),
    "quantity": rng.integers(low=1, high=6, size=n),
    "region": rng.choice(["Moscow", "Saint Petersburg", "Kazan", "Novosibirsk"], size=n),
    "channel": rng.choice(["Online", "Retail", "Online"], size=n),
})

# Вводим пропуски и дубликат
sales_df.loc[3, "amount"] = np.nan
sales_df.loc[7, "client"] = np.nan
sales_df.loc[10, "quantity"] = np.nan
sales_df = pd.concat([sales_df, sales_df.iloc[[0]]], ignore_index=True)

print("shape:", sales_df.shape)
sales_df.head()


**Что делает код:**

- `np.random.default_rng(seed=42)` — генератор с фиксированным зерном для воспроизводимости
- `pd.date_range(..., freq="D")` — диапазон дней с шагом 1
- `rng.choice(...)` — выборка значений из списка
- `rng.normal(loc=120, scale=40)` — нормальное распределение
- `np.round(..., 2)` — округление до двух знаков (для денежных сумм)


## 2. Подготовка файлов для примеров чтения

Чтобы продемонстрировать чтение из разных форматов, сначала сохраним `sales_df`
в Excel, CSV и Parquet в папку `data/`.


In [ ]:
import os
os.makedirs("data", exist_ok=True)

sales_df.to_excel("data/sales.xlsx", index=False, sheet_name="Sales 2024")
sales_df.to_csv("data/sales.csv", index=False)
sales_df.to_parquet("data/sales.parquet", index=False)

# С русским разделителем ; и кодировкой cp1251
sales_df.to_csv("data/sales_ru.csv", sep=";", index=False, encoding="utf-8-sig")

print("Файлы созданы:", os.listdir("data"))


## 2.1. Чтение из локальных файлов

Все три формата читаются одной строкой, но параметры сильно влияют на результат.


In [ ]:
# Чтение Excel — выбор листа по имени
df_xlsx = pd.read_excel("data/sales.xlsx", sheet_name="Sales 2024")
print("Excel:", df_xlsx.shape)
print(df_xlsx.dtypes)
print()

# Чтение CSV — по умолчанию запятая и UTF-8
df_csv = pd.read_csv("data/sales.csv")
print("CSV:", df_csv.shape)

# Чтение CSV с русским разделителем
df_csv_ru = pd.read_csv("data/sales_ru.csv", sep=";", encoding="utf-8-sig")
print("CSV RU:", df_csv_ru.shape)

# Чтение Parquet — типы сохраняются
df_pq = pd.read_parquet("data/sales.parquet")
print("Parquet:", df_pq.shape)
print(df_pq.dtypes)


## 2.2. Чтение по прямой ссылке (демонстрация)

Если бы файл лежал в открытом GitHub-репозитории, можно было бы читать его
напрямую. Ниже — примеры с шаблонными ссылками `USERNAME/REPOSITORY`.


In [ ]:
# Примеры шаблонных ссылок (замените USERNAME и REPOSITORY на свои)
base = "https://raw.githubusercontent.com/USERNAME/REPOSITORY/main/data"

# Раскомментируйте и подставьте свой репозиторий:
# df_url = pd.read_csv(f"{base}/sales.csv")
# df_pq_url = pd.read_parquet(f"{base}/sales.parquet")

print("Шаблон URL для чтения:", base + "/sales.csv")


## 3. Первичный осмотр и полезные команды

После чтения данных всегда выполняйте быстрый осмотр. Это помогает заметить
проблемы до того, как они станут критичными.


In [ ]:
# Размер и форма
print("shape:", sales_df.shape)
print("rows:", len(sales_df))
print("cols:", sales_df.columns.tolist())

# Общая информация о типах и памяти
sales_df.info()

# Использование памяти с учётом реальных типов
print(sales_df.memory_usage(deep=True))


In [ ]:
# Первые и последние строки
sales_df.head()


In [ ]:
sales_df.tail()


In [ ]:
# Случайная выборка — для проверки, что данные перемешаны
sales_df.sample(5, random_state=42)


In [ ]:
# Типы столбцов
print(sales_df.dtypes)
print()
print(sales_df.dtypes.value_counts())

# Выбор столбцов определённого типа
numeric_cols = sales_df.select_dtypes(include="number").columns
text_cols = sales_df.select_dtypes(include="object").columns
date_cols = sales_df.select_dtypes(include="datetime").columns

print("Числовые:", numeric_cols.tolist())
print("Текстовые:", text_cols.tolist())
print("Даты:", date_cols.tolist())


In [ ]:
# Описательная статистика
sales_df.describe()


In [ ]:
# Включая нечисловые столбцы
sales_df.describe(include="all")


In [ ]:
# Пропуски
print(sales_df.isna().sum())
print()
print("Доля пропусков:")
print((sales_df.isna().mean().sort_values(ascending=False).round(3) * 100))


In [ ]:
# Уникальные значения
print(sales_df.nunique())
print()
print("Частоты category:")
print(sales_df["category"].value_counts(dropna=False))
print()
print("Частоты region:")
print(sales_df["region"].value_counts(normalize=True).round(3))


### Шаблон быстрого осмотра

Функцию `quick_inspect` можно скопировать в любой проект. Она даёт полный
снимок данных за один вызов.


In [ ]:
def quick_inspect(df: pd.DataFrame, name: str = "df") -> None:
    '''Быстрый осмотр DataFrame после чтения.'''
    print(f"=== {name} ===")
    print(f"shape: {df.shape}")
    print(f"\ndtypes:\n{df.dtypes}")
    print(f"\nПервые строки:\n{df.head()}")
    print(f"\nОписательная статистика:\n{df.describe(include='all')}")
    print(f"\nПропуски по столбцам:\n{df.isna().sum()}")
    print(f"\nУникальные значения:\n{df.nunique()}")

quick_inspect(sales_df, "sales_df")


## 4. Анализ и преобразование данных

Дисциплина: `df_raw` — не трогаем, `df_work` — рабочая копия, `df_clean` — финал.


In [ ]:
# Зафиксируем исходник
df_raw = sales_df.copy()

# Рабочая копия
df_work = df_raw.copy()

# Шаг 1: удаление дубликатов
print("До:", len(df_work))
df_work = df_work.drop_duplicates()
print("После drop_duplicates:", len(df_work))


In [ ]:
# Шаг 2: заполнение пропусков
median_amount = df_work["amount"].median()
df_work["amount"] = df_work["amount"].fillna(median_amount)
df_work["client"] = df_work["client"].fillna("Unknown")
df_work["quantity"] = df_work["quantity"].fillna(1).astype(int)

# Проверка
print(df_work.isna().sum())


In [ ]:
# Шаг 3: преобразование даты
df_work["order_date"] = pd.to_datetime(df_work["order_date"])
df_work["year"] = df_work["order_date"].dt.year
df_work["month"] = df_work["order_date"].dt.month
df_work["weekday"] = df_work["order_date"].dt.day_name()
df_work["is_weekend"] = df_work["order_date"].dt.weekday >= 5

print(df_work[["order_date", "year", "month", "weekday", "is_weekend"]].head())


In [ ]:
# Шаг 4: очистка текста
df_work["client_clean"] = df_work["client"].str.strip()
df_work["category_clean"] = df_work["category"].str.strip().str.lower()

print(df_work["category_clean"].value_counts())


In [ ]:
# Шаг 5: поиск выбросов через IQR
q1 = df_work["amount"].quantile(0.25)
q3 = df_work["amount"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f"IQR границы: [{lower:.2f}, {upper:.2f}]")
mask_outliers = (df_work["amount"] < lower) | (df_work["amount"] > upper)
print("Выбросов:", mask_outliers.sum())

# Ограничение выбросов (clip) — не удаляем, а прижимаем к границам
df_work["amount"] = df_work["amount"].clip(lower=lower, upper=upper)


### Агрегации и groupby


In [ ]:
# Простая агрегация
print(df_work.groupby("region")["amount"].sum())
print()

# Несколько функций
df_work.groupby("region")["amount"].agg(["sum", "mean", "count"])


In [ ]:
# Разные функции к разным столбцам с переименованием
df_work.groupby("region").agg(
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean"),
    n_orders=("order_id", "count"),
    n_clients=("client", "nunique"),
)


In [ ]:
# Группировка по нескольким столбцам
df_work.groupby(["region", "category"], as_index=False)["amount"].sum()


### Сводные таблицы


In [ ]:
# Сводная таблица: регион × категория, сумма amount
pivot_sum = df_work.pivot_table(
    values="amount",
    index="region",
    columns="category",
    aggfunc="sum",
    fill_value=0,
)
pivot_sum


In [ ]:
# С margins — итоги по строкам и столбцам
pivot_margins = df_work.pivot_table(
    values="amount",
    index="region",
    columns="category",
    aggfunc="sum",
    margins=True,
    margins_name="Итого",
)
pivot_margins


### transform: агрегация с сохранением формы


In [ ]:
# Средний чек по региону в каждой строке
df_work["region_avg"] = df_work.groupby("region")["amount"].transform("mean")

# Доля суммы заказа от суммы по региону
df_work["share_of_region"] = df_work["amount"] / df_work.groupby("region")["amount"].transform("sum")

print(df_work[["order_id", "region", "amount", "region_avg", "share_of_region"]].head())


### Объединение наборов


In [ ]:
# Создадим справочник клиентов
clients_df = pd.DataFrame({
    "client": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "city": ["Moscow", "Saint Petersburg", "Kazan", "Novosibirsk", "Moscow"],
    "tier": ["Gold", "Silver", "Silver", "Bronze", "Gold"],
})

# merge по ключу
df_merged = df_work.merge(clients_df, on="client", how="left")
print("Было:", df_work.shape, "Стало:", df_merged.shape)
print("Новые столбцы:", set(df_merged.columns) - set(df_work.columns))
df_merged.head()


## 5. Целостность и безопасность данных

Дисциплина `df_raw` / `df_work` / `df_clean` + аудит на каждом шаге.


In [ ]:
# Снимок состояния DataFrame для последующего сравнения
def snapshot(df: pd.DataFrame, name: str = "") -> dict:
    '''Снимок состояния DataFrame.'''
    return {
        "name": name,
        "rows": len(df),
        "cols": df.shape[1],
        "amount_sum": df["amount"].sum() if "amount" in df.columns else None,
        "amount_mean": df["amount"].mean() if "amount" in df.columns else None,
        "n_na_amount": df["amount"].isna().sum() if "amount" in df.columns else None,
    }

before = snapshot(df_work, "before")
print("Before:", before)


In [ ]:
def audit(df: pd.DataFrame, step_name: str) -> None:
    '''Контроль состояния DataFrame после шага.'''
    print(f"\n=== {step_name} ===")
    print(f"shape: {df.shape}")
    print(f"dtypes:\n{df.dtypes}")
    print(f"na sum:\n{df.isna().sum()}")

audit(df_raw, "Шаг 0: исходный")


In [ ]:
# Полный сценарий очистки с аудитом
df_work = df_raw.copy()
audit(df_work, "Шаг 0: исходный")

df_work = df_work.drop_duplicates()
audit(df_work, "Шаг 1: drop_duplicates")

df_work["amount"] = df_work["amount"].fillna(df_work["amount"].median())
df_work["client"] = df_work["client"].fillna("Unknown")
df_work["quantity"] = df_work["quantity"].fillna(1).astype(int)
audit(df_work, "Шаг 2: fillna")

df_work["order_date"] = pd.to_datetime(df_work["order_date"])
audit(df_work, "Шаг 3: to_datetime")

df_clean = df_work.copy()
audit(df_clean, "Шаг 4: df_clean")
print(f"\n=== ИТОГО ===")
print(f"Исходник: {df_raw.shape}, Чистый: {df_clean.shape}")


### Защита от потери типов при CSV round-trip


In [ ]:
# Сохраним во всех трёх форматах и сравним типы после перечитывания
df_clean.to_csv("data/check.csv", index=False)
df_clean.to_parquet("data/check.parquet", index=False)

df_csv_back = pd.read_csv("data/check.csv")
df_pq_back = pd.read_parquet("data/check.parquet")

print("Исходные типы:")
print(df_clean.dtypes)
print("\nПосле CSV round-trip:")
print(df_csv_back.dtypes)
print("\nПосле Parquet round-trip:")
print(df_pq_back.dtypes)
print("\nParquet сохранил типы:", df_clean.dtypes.equals(df_pq_back.dtypes))


## 6. Визуализация

Пять ключевых типов графиков: столбчатая диаграмма, гистограмма, boxplot,
линейный график, тепловая карта корреляций.


In [ ]:
import os
os.makedirs("figures", exist_ok=True)


### Столбчатая диаграмма


In [ ]:
region_sum = df_clean.groupby("region")["amount"].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=region_sum, x="region", y="amount", ax=ax, color="#1F3A5F")
ax.set_title("Сумма заказов по регионам", fontsize=14)
ax.set_xlabel("Регион")
ax.set_ylabel("Сумма, у.е.")
ax.tick_params(axis="x", rotation=20)

fig.savefig("figures/bar_region.png", dpi=150, bbox_inches="tight")
plt.show()


### Гистограмма


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df_clean, x="amount", bins=20, kde=True, color="#2F5A8F", ax=ax)
ax.set_title("Распределение суммы заказов", fontsize=14)
ax.set_xlabel("Сумма заказа")
ax.set_ylabel("Число заказов")
ax.axvline(df_clean["amount"].mean(), color="red", linestyle="--",
           label=f"Среднее: {df_clean['amount'].mean():.1f}")
ax.axvline(df_clean["amount"].median(), color="green", linestyle="--",
           label=f"Медиана: {df_clean['amount'].median():.1f}")
ax.legend()

fig.savefig("figures/hist_amount.png", dpi=150, bbox_inches="tight")
plt.show()


### Boxplot


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_clean, x="category", y="amount", ax=ax, color="#1F3A5F")
ax.set_title("Распределение суммы по категориям", fontsize=14)
ax.set_xlabel("Категория")
ax.set_ylabel("Сумма заказа")

medians = df_clean.groupby("category")["amount"].median()
for i, cat in enumerate(medians.index):
    ax.text(i, medians[cat], f"{medians[cat]:.0f}",
            ha="center", va="bottom", fontsize=10, color="red")

fig.savefig("figures/boxplot_category.png", dpi=150, bbox_inches="tight")
plt.show()


### Линейный график


In [ ]:
daily = df_clean.groupby("order_date")["amount"].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=daily, x="order_date", y="amount", ax=ax, color="#1F3A5F", marker="o")
ax.set_title("Динамика суммы заказов по дням", fontsize=14)
ax.set_xlabel("Дата")
ax.set_ylabel("Сумма, у.е.")
ax.tick_params(axis="x", rotation=30)

daily["rolling_7"] = daily["amount"].rolling(window=7, min_periods=1).mean()
ax.plot(daily["order_date"], daily["rolling_7"], color="#B25435",
        linewidth=2, label="Скользящее 7д")
ax.legend()

fig.savefig("figures/line_daily.png", dpi=150, bbox_inches="tight")
plt.show()


### Тепловая карта корреляций


In [ ]:
numeric_df = df_clean.select_dtypes(include="number")
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Корреляция числовых признаков", fontsize=14)
fig.savefig("figures/heatmap_corr.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Сохранение результатов

Сохраняем в трёх форматах и проверяем, что файлы создались корректно.


In [ ]:
import os

os.makedirs("output", exist_ok=True)

# Excel
df_clean.to_excel("output/sales_clean.xlsx", sheet_name="Sales 2024", index=False)

# CSV (UTF-8)
df_clean.to_csv("output/sales_clean.csv", index=False)

# CSV с русским разделителем
df_clean.to_csv("output/sales_ru.csv", sep=";", decimal=",",
                 index=False, encoding="utf-8-sig")

# Parquet
df_clean.to_parquet("output/sales_clean.parquet", index=False)

# Несколько листов в один Excel
with pd.ExcelWriter("output/sales_multi.xlsx") as writer:
    df_clean.to_excel(writer, sheet_name="Clean", index=False)
    df_work.to_excel(writer, sheet_name="Work", index=False)
    clients_df.to_excel(writer, sheet_name="Clients", index=False)

print("Файлы созданы:")
for f in sorted(os.listdir("output")):
    size = os.path.getsize(os.path.join("output", f))
    print(f"  {f}: {size:,} байт")


### Проверка сохранённых файлов


In [ ]:
# Проверка, что файлы существуют и перечитываются
path = "output/sales_clean.csv"
print("Файл существует:", os.path.exists(path))
print("Размер, байт:", os.path.getsize(path))

df_check = pd.read_csv(path)
print("Совпадение форм:", df_check.shape == df_clean.shape)
print("Число строк:", len(df_check), "==", len(df_clean))

# Parquet — типы сохраняются
df_pq_check = pd.read_parquet("output/sales_clean.parquet")
print("\nParquet типы совпадают:", df_pq_check.dtypes.equals(df_clean.dtypes))


## 8. Итоговый чек-лист

Перед завершением работы проверьте, что все шаги выполнены:

- [ ] Все ячейки выполнены без ошибок (`Kernel → Restart & Run All`)
- [ ] Воспроизводимый `sales_df` генерируется в первой ячейке после импортов
- [ ] Используется дисциплина `df_raw` / `df_work` / `df_clean`
- [ ] После каждого важного преобразования есть проверка результата
- [ ] Графики сохранены в папку `figures/`
- [ ] Результаты сохранены в трёх форматах в папку `output/`
- [ ] Файлы перечитаны и проверены

## Полезные ссылки

- Документация pandas: https://pandas.pydata.org/docs/
- Документация openpyxl: https://openpyxl.readthedocs.io/
- Документация pyarrow: https://arrow.apache.org/docs/python/
- Документация seaborn: https://seaborn.pydata.org/
